<a href="https://colab.research.google.com/github/SpyC0der77/vscolab/blob/master/vscolab_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""Colab AI bridge for the vscolab Language Model Chat Provider extension.

Starts a localhost HTTP server that wraps ``google.colab.ai`` so the VS Code
extension (running in openvscode-server) can call Gemini without an API key.
"""

from __future__ import annotations

import json
import os
import socket
import threading
import time
import traceback
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from typing import Any, Iterator
from urllib.parse import urlparse

BRIDGE_HOST = "127.0.0.1"
BRIDGE_PORT = 8787
DEFAULT_MODELS = [
    "gemini-3.5-flash",
    "gemini-3.1-pro",
    "gemini-3.0-flash",
    "gemini-2.5-flash",
]
DEFAULT_MODEL = DEFAULT_MODELS[0]

# Notebook cells re-exec `_server = None` on every run. Keep the live server on a
# stable key so re-runs can shut it down without killing the Colab kernel.
_SERVER_KEY = "_vscolab_lm_http_server"


def _get_server() -> ThreadingHTTPServer | None:
    return globals().get(_SERVER_KEY)


def _set_server(server: ThreadingHTTPServer | None) -> None:
    globals()[_SERVER_KEY] = server


class _ReuseHTTPServer(ThreadingHTTPServer):
    allow_reuse_address = True
    daemon_threads = True


def _bridge_healthy(host: str, port: int) -> bool:
    try:
        with urllib.request.urlopen(f"http://{host}:{port}/health", timeout=1) as resp:
            return resp.status == 200
    except Exception:
        return False


def _request_shutdown(host: str, port: int) -> None:
    """Ask a live bridge (possibly from a prior cell run) to stop itself."""
    try:
        req = urllib.request.Request(
            f"http://{host}:{port}/shutdown",
            method="POST",
            data=b"{}",
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=2):
            pass
    except Exception:
        pass


def _stop_server(server: ThreadingHTTPServer | None) -> None:
    if server is None:
        return
    try:
        server.shutdown()
    except Exception:
        pass
    try:
        server.server_close()
    except Exception:
        pass


def _wait_port_free(host: str, port: int, timeout: float = 3.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(0.2)
            if sock.connect_ex((host, port)) != 0:
                return True
        time.sleep(0.1)
    return False


def _list_models() -> list[str]:
    discovered: list[str] = []
    try:
        from google.colab import ai

        discovered = [str(m) for m in ai.list_models()]
    except Exception as exc:
        print(f"colab_lm_bridge: list_models failed ({exc}); using defaults", flush=True)

    # Always surface the curated Gemini models, then any extras from Colab.
    seen: set[str] = set()
    models: list[str] = []
    for mid in [*DEFAULT_MODELS, *discovered]:
        if mid in seen:
            continue
        seen.add(mid)
        models.append(mid)
    return models or list(DEFAULT_MODELS)


def _generate(prompt: str, model: str | None, stream: bool) -> str | Iterator[str]:
    from google.colab import ai

    kwargs: dict[str, Any] = {"stream": stream}
    if model:
        # Colab has used both names across versions; prefer model_name (current runtime).
        try:
            return ai.generate_text(prompt, model_name=model, **kwargs)
        except TypeError as exc:
            if "model_name" not in str(exc) and "unexpected keyword" not in str(exc):
                raise
            return ai.generate_text(prompt, model=model, **kwargs)
    return ai.generate_text(prompt, **kwargs)


def _messages_to_prompt(messages: list[dict[str, str]]) -> str:
    parts: list[str] = []
    for msg in messages:
        role = msg.get("role") or "user"
        content = msg.get("content") or ""
        if role == "assistant":
            parts.append(f"Assistant: {content}")
        else:
            parts.append(f"User: {content}")
    parts.append("Assistant:")
    return "\n\n".join(parts)


class _BridgeHandler(BaseHTTPRequestHandler):
    server_version = "vscolab-colab-lm/1.0"

    def log_message(self, fmt: str, *args: Any) -> None:
        print(f"colab_lm_bridge: {fmt % args}", flush=True)

    def _send(self, code: int, body: bytes, content_type: str) -> None:
        self.send_response(code)
        self.send_header("Content-Type", content_type)
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(body)

    def _json(self, code: int, obj: Any) -> None:
        self._send(code, json.dumps(obj).encode(), "application/json")

    def do_OPTIONS(self) -> None:  # noqa: N802
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type")
        self.end_headers()

    def do_GET(self) -> None:  # noqa: N802
        path = urlparse(self.path).path.rstrip("/") or "/"
        if path == "/health":
            self._json(200, {"status": "ok", "pid": os.getpid()})
            return
        if path == "/models":
            self._json(200, {"models": _list_models()})
            return
        self._json(404, {"error": "not found"})

    def do_POST(self) -> None:  # noqa: N802
        path = urlparse(self.path).path.rstrip("/") or "/"
        if path == "/shutdown":
            self._json(200, {"status": "shutting_down"})
            server = self.server

            def _stop() -> None:
                time.sleep(0.05)
                try:
                    server.shutdown()
                except Exception:
                    pass
                try:
                    server.server_close()
                except Exception:
                    pass

            threading.Thread(target=_stop, daemon=True).start()
            return

        if path != "/generate":
            self._json(404, {"error": "not found"})
            return

        length = int(self.headers.get("Content-Length", 0))
        body = self.rfile.read(length) if length else b"{}"
        try:
            payload = json.loads(body.decode() or "{}")
        except json.JSONDecodeError:
            self._json(400, {"error": "invalid json"})
            return

        prompt = payload.get("prompt")
        messages = payload.get("messages")
        if not prompt and messages:
            prompt = _messages_to_prompt(messages)
        if not prompt:
            self._json(400, {"error": "prompt or messages required"})
            return

        model = payload.get("model")
        stream = bool(payload.get("stream"))

        try:
            result = _generate(prompt, model, stream)
        except Exception as exc:
            print(f"colab_lm_bridge: generate failed: {exc}", flush=True)
            traceback.print_exc()
            self._json(500, {"error": str(exc)})
            return

        if not stream:
            self._json(200, {"text": str(result)})
            return

        self.send_response(200)
        self.send_header("Content-Type", "application/x-ndjson")
        self.send_header("Transfer-Encoding", "chunked")
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()

        try:
            for chunk in result:
                line = json.dumps({"text": str(chunk)}) + "\n"
                data = line.encode()
                self.wfile.write(f"{len(data):x}\r\n".encode())
                self.wfile.write(data)
                self.wfile.write(b"\r\n")
                self.wfile.flush()
            self.wfile.write(b"0\r\n\r\n")
        except Exception as exc:
            print(f"colab_lm_bridge: stream failed: {exc}", flush=True)


def setup_colab_lm(host: str = BRIDGE_HOST, port: int = BRIDGE_PORT) -> str:
    """Start the Colab AI bridge and return its base URL.

    Restarts the in-process server on notebook re-runs. Never kills the Colab
    kernel (the bridge shares this process).
    """
    url = f"http://{host}:{port}"

    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError(
            "google.colab.ai is only available inside Google Colab."
        ) from exc

    existing = _get_server()
    if existing is not None:
        print("Stopping previous Colab AI bridge...", flush=True)
        _stop_server(existing)
        _set_server(None)
        _wait_port_free(host, port)
    elif _bridge_healthy(host, port):
        # Prior cell left a live bridge but lost the Python reference.
        print("Stopping previous Colab AI bridge via /shutdown...", flush=True)
        _request_shutdown(host, port)
        _wait_port_free(host, port)

    if _bridge_healthy(host, port):
        # Still serving (shutdown raced or foreign process). Reuse — do NOT
        # fuser/kill: that would terminate this Colab kernel.
        print(f"Colab AI bridge already listening at {url} (reusing)", flush=True)
        return url

    try:
        server = _ReuseHTTPServer((host, port), _BridgeHandler)
    except OSError as exc:
        if getattr(exc, "errno", None) != 98:  # EADDRINUSE
            raise
        if _bridge_healthy(host, port):
            print(f"Colab AI bridge already listening at {url} (reusing)", flush=True)
            return url
        raise RuntimeError(
            f"Port {port} is busy and the existing Colab AI bridge did not respond. "
            "Restart the Colab runtime, then re-run this cell."
        ) from exc

    _set_server(server)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()

    print(f"Colab AI bridge listening at {url}", flush=True)
    return url

"""Official Microsoft VS Code bootstrap via Remote Tunnels (``code tunnel``).

Colab's ``proxyPort`` cannot reliably upgrade WebSockets for Microsoft's web
server (browser error 1006). The supported path is the VS Code CLI tunnel,
which opens ``https://vscode.dev/tunnel/...`` (or desktop Remote Tunnels).
"""

from __future__ import annotations

import os
import re
import subprocess
import time
from pathlib import Path

CLI_DOWNLOAD = (
    "https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64"
)

# Colab has no desktop keyring; force file-backed credential storage.
os.environ.setdefault("VSCODE_CLI_USE_FILE_KEYCHAIN", "1")


def _download(url: str, dest: Path, label: str) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Using cached {label} at {dest}", flush=True)
        return
    print(f"Downloading {label}...", flush=True)
    subprocess.run(
        ["wget", "--show-progress", "-O", str(dest), url],
        check=True,
    )


def _is_valid_vsix(path: Path) -> bool:
    with path.open("rb") as f:
        return f.read(2) == b"PK"


def _ensure_vsix(vsix_path: Path, url: str, label: str) -> None:
    if vsix_path.exists() and _is_valid_vsix(vsix_path):
        return
    if vsix_path.exists():
        print(f"Removing invalid VSIX at {vsix_path}", flush=True)
        vsix_path.unlink()
    _download(url, vsix_path, label)
    if not _is_valid_vsix(vsix_path):
        raise RuntimeError(
            f"Downloaded file at {vsix_path} is not a valid VSIX (expected zip archive)."
        )


def prepare_vscode(
    cache_dir: Path,
    user_data_dir: Path,
    commit: str = "",
) -> dict:
    """Download the VS Code CLI. ``commit`` is unused (CLI tracks stable)."""
    del commit  # kept for call-site compatibility with older notebooks
    cache_dir = Path(cache_dir).resolve()
    user_data_dir = Path(user_data_dir).resolve()
    cache_dir.mkdir(parents=True, exist_ok=True)
    user_data_dir.mkdir(parents=True, exist_ok=True)

    cli_data_dir = user_data_dir / "cli"
    cli_data_dir.mkdir(parents=True, exist_ok=True)

    code_bin = cache_dir / "code"
    if not code_bin.exists():
        tarball = cache_dir / "vscode_cli_alpine_x64_cli.tar.gz"
        _download(CLI_DOWNLOAD, tarball, "VS Code CLI")
        print("Extracting VS Code CLI...", flush=True)
        subprocess.run(
            ["tar", "-xzf", str(tarball), "-C", str(cache_dir)],
            check=True,
        )
        if not code_bin.exists():
            found = [p for p in cache_dir.rglob("code") if p.is_file()]
            if not found:
                raise RuntimeError(f"VS Code CLI binary not found under {cache_dir}")
            if found[0] != code_bin:
                found[0].replace(code_bin)
        code_bin.chmod(0o755)

    print(f"VS Code CLI ready at {code_bin}", flush=True)
    return {
        "code_bin": code_bin,
        "cache_dir": cache_dir,
        "user_data_dir": user_data_dir,
        "cli_data_dir": cli_data_dir,
        "server_bin": code_bin,
        "extensions_dir": user_data_dir / "extensions",
    }


def _cli_base(prepared: dict) -> list[str]:
    return [
        str(prepared["code_bin"]),
        "--cli-data-dir",
        str(prepared["cli_data_dir"]),
    ]


def login_vscode(prepared: dict, provider: str = "github", timeout: int = 600) -> None:
    """Run GitHub/Microsoft device login for the tunnel (interactive once)."""
    cmd = [
        *_cli_base(prepared),
        "tunnel",
        "user",
        "login",
        "--provider",
        provider,
    ]
    print(f"Starting VS Code tunnel login ({provider})...", flush=True)
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if proc.stdout is None:
        raise RuntimeError("Failed to capture login process output.")

    url_re = re.compile(
        r"(https://(?:github\.com/login/device|login\.microsoftonline\.com/\S+))"
    )
    code_re = re.compile(r"(?:use code|enter the code)\s+([A-Z0-9-]+)", re.I)
    code_re2 = re.compile(r"\b([A-Z0-9]{4,}-[A-Z0-9]{4,})\b")

    shown_auth = False
    deadline = time.time() + timeout
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line and proc.poll() is not None:
            break
        if not line:
            time.sleep(0.05)
            continue
        line = line.rstrip()
        print(line, flush=True)

        url_m = url_re.search(line)
        code_m = code_re.search(line) or code_re2.search(line)
        if url_m and code_m and not shown_auth:
            print(
                f"\n>>> Authorize this Colab session:\n"
                f"    Open {url_m.group(1)}\n"
                f"    Enter code: {code_m.group(1)}\n",
                flush=True,
            )
            shown_auth = True

    if proc.poll() is None:
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            pass

    print("Tunnel login finished.", flush=True)


def _extension_install_args(extensions: list, cache_dir: Path) -> list[str]:
    """Build repeated ``--install-extension`` args (marketplace IDs or VSIX paths)."""
    args: list[str] = []
    for ext in extensions or []:
        if isinstance(ext, str):
            args.extend(["--install-extension", ext])
            continue
        vsix = ext["vsix"]
        vsix_path = Path(vsix)
        if not vsix_path.is_absolute():
            vsix_path = cache_dir / vsix
        if ext.get("url"):
            _ensure_vsix(vsix_path, ext["url"], ext.get("id") or vsix_path.name)
        elif not vsix_path.exists() or not _is_valid_vsix(vsix_path):
            raise FileNotFoundError(f"VSIX not found: {vsix_path}")
        args.extend(["--install-extension", str(vsix_path)])
    return args


def start_vscode_web(
    prepared: dict,
    folder: str | Path,
    port: int = 3000,
    host: str = "0.0.0.0",
    *,
    tunnel_name: str = "vscolab",
    extensions: list | None = None,
    timeout: int = 180,
) -> str:
    """Start ``code tunnel`` and return the ``vscode.dev`` URL.

    ``port`` / ``host`` are ignored (kept for call-site compatibility).
    """
    del port, host
    folder = str(Path(folder).resolve())
    Path(folder).mkdir(parents=True, exist_ok=True)

    cmd = [
        *_cli_base(prepared),
        "tunnel",
        "--accept-server-license-terms",
        "--name",
        tunnel_name,
        *_extension_install_args(extensions or [], prepared["cache_dir"]),
    ]

    print(f"Starting VS Code tunnel '{tunnel_name}' (cwd={folder})...", flush=True)
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=folder,
    )
    if proc.stdout is None:
        raise RuntimeError("Failed to capture tunnel process output.")

    url_re = re.compile(r"(https://vscode\.dev/tunnel/[^\s]+)")
    deadline = time.time() + timeout
    tunnel_url: str | None = None

    while time.time() < deadline:
        if proc.poll() is not None and tunnel_url is None:
            rest = proc.stdout.read() or ""
            if rest:
                print(rest, end="", flush=True)
            raise RuntimeError(
                f"VS Code tunnel exited early (code {proc.returncode}) before URL appeared."
            )
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.05)
            continue
        print(line.rstrip(), flush=True)
        match = url_re.search(line)
        if match:
            tunnel_url = match.group(1).rstrip(").,]")
            break

    if not tunnel_url:
        raise RuntimeError(
            f"Timed out waiting for vscode.dev tunnel URL ({timeout}s)."
        )

    folder_name = Path(folder).name
    if folder_name and not tunnel_url.rstrip("/").endswith(folder_name):
        tunnel_url = tunnel_url.rstrip("/") + "/" + folder_name

    print(f"VS Code tunnel ready: {tunnel_url}", flush=True)
    prepared["tunnel_proc"] = proc
    return tunnel_url


def vscode_proxy_url(base_proxy_url: str, folder: str | Path) -> str:
    """Compatibility shim — tunnel URLs are already complete."""
    del folder
    return base_proxy_url

from pathlib import Path


PORT = 3000  # unused with tunnels; kept for notebook compatibility
GIT_REPO = ""
COMMIT = ""
VSCOLAB_RAW = "https://github.com/SpyC0der77/vscolab/raw/master"
EXTENSIONS = [
    # Copilot Chat ships with official VS Code — do not marketplace-install it.
    {
        "vsix": "colab-lm-0.2.0.vsix",
        "url": f"{VSCOLAB_RAW}/extensions/colab-lm/colab-lm-0.2.0.vsix",
    },
]
CACHE_DIR = Path("/content/vscode-cache")
USER_DATA_DIR = Path("/content/.vscode-server-data")
TUNNEL_NAME = "vscolab-ai"

folder = Path("/content/workspace")
if GIT_REPO:
    import subprocess

    name = GIT_REPO.rstrip("/").removesuffix(".git").split("/")[-1]
    folder = Path(f"/content/{name}")
    if not folder.exists():
        print(f"Cloning {GIT_REPO}...", flush=True)
        subprocess.run(["git", "clone", "--progress", GIT_REPO, str(folder)], check=True)
    else:
        print(f"Using existing clone at {folder}", flush=True)
else:
    folder.mkdir(parents=True, exist_ok=True)

folder = str(folder.resolve())
prepared = prepare_vscode(CACHE_DIR, USER_DATA_DIR, COMMIT)
login_vscode(prepared)
setup_colab_lm()
url = start_vscode_web(
    prepared,
    folder,
    tunnel_name=TUNNEL_NAME,
    extensions=EXTENSIONS,
)
print(f"Open VS Code: {url}", flush=True)
print("Colab AI bridge is on http://127.0.0.1:8787 (reachable from the remote tunnel).", flush=True)